# Creación y ajuste del dataset del TFM

## Razonamiento:

Tras haber realizado un merge global con todos los datasets, hemos comprobado que hay varios sets donde la cantidad de usuarios varía y dejan de ser los mismos sujetos que en otros. Esto ha sólo aquellos datasets con mayor concetración de pacientes sobrevivan y la información de otros muy relevantes al asunto clave del TFM se pierda.

Por lo tanto, previo a un merge global se va a realizar uno agrupando los sets en las categorias de:

- Merge Examination
- Merge Lab
- Merge Questionary
- Merge Demography

**Clave post-merge**

Tras realizar el merge entre los datasets de una misma categoría se han eliminado aquellas variables cuyo porcentage de nulos superaba el 30% para asegurar una integridad mínima en las variables previas al estudio.

In [4]:
#!pip install --upgrade pandas numpy

import pandas as pd
import os

### Merge Examination data

In [3]:
# 1. Acceder a la carpeta y cargar los archivos
folder_path = 'data/'
files = ["BAX_L.xpt", "BPXO_L.xpt", "BMX_L.xpt", "LUX_L.xpt"]

datasets = []
for file in files:
    path = os.path.join(folder_path, file)
    # Cargamos el archivo XPT
    df = pd.read_sas(path)
    datasets.append(df)

# 2 y 3. Revisar SEQN y hacer merge únicamente con coincidencias en todos
# Iniciamos el merge con el primer dataframe
merged_df = datasets[0]

# Realizamos un "inner join" sucesivo con el resto
for df in datasets[1:]:
    merged_df = pd.merge(merged_df, df, on='SEQN', how='inner')

# 4. Análisis de nulos
# Nulos por columna
nulls_per_column = merged_df.isnull().sum()
# Nulos por fila
nulls_per_row = merged_df.isnull().sum(axis=1)

print("--- Análisis de Nulos por Columna (Primeras 10) ---")
print(nulls_per_column.head(10))

print("\n--- Resumen de Nulos por Fila ---")
print(nulls_per_row.describe())

# 5. Salvar el merge
# Lo guardamos en formato CSV para fácil lectura posterior
merged_df.to_csv('data/merged_sets/merged_examination_data.csv', index=False)

print("\nProceso completado. Archivo guardado como 'merged_examination_data.csv'")

--- Análisis de Nulos por Columna (Primeras 10) ---
SEQN           0
BAXMSTAT       0
BAXRXNC     4567
BAXRXND     4558
BAX5STAT     206
BAQ110       408
BAQ121       466
BAQ125       492
BAQ132       747
BAQ140       747
dtype: int64

--- Resumen de Nulos por Fila ---
count    4771.000000
mean       39.563613
std         9.312414
min        23.000000
25%        34.000000
50%        37.000000
75%        41.000000
max        80.000000
dtype: float64

Proceso completado. Archivo guardado como 'merged_examination_data.csv'


In [4]:
# 1. Cargar el merge generado anteriormente
df = pd.read_csv('merged_examination_data.csv')

# 2. Definir el umbral (e.g., conservar columnas con al menos 70% de datos)
umbral_porcentaje = 0.7
min_datos_requeridos = int(umbral_porcentaje * len(df))

# 3. Aplicar la limpieza
# axis=1 indica que eliminamos columnas. thresh pide el mínimo de valores NO nulos.
df_clean = df.dropna(axis=1, thresh=min_datos_requeridos)

print(f"Columnas originales: {df.shape[1]}")
print(f"Columnas después de limpieza: {df_clean.shape[1]}")
print(f"Variables eliminadas por exceso de nulos: {df.shape[1] - df_clean.shape[1]}")

# 4. Guardar el dataset optimizado
df_clean.to_csv('data/merged_sets/merged_examination_data_clean.csv', index=False)

# Mostrar las primeras 5 columnas restantes para verificar
print("\nPrimeras columnas del dataset limpio:")
print(df_clean.columns.tolist()[:10])

Columnas originales: 89
Columnas después de limpieza: 49
Variables eliminadas por exceso de nulos: 40

Primeras columnas del dataset limpio:
['SEQN', 'BAXMSTAT', 'BAX5STAT', 'BAQ110', 'BAQ121', 'BAQ125', 'BAQ132', 'BAQ140', 'BAQ150', 'BAQ160']


## Merge Lab Data

In [8]:
# 1. Acceder a la carpeta y cargar los archivos
folder_path = 'data/'
files = ["ALB_CR_L.xpt", "CBC_L.xpt", "TRIGLY_L.xpt", 
         "HDL_L.xpt", "HSCRP_L.xpt", "PBCD_L.xpt"]

datasets = []
for file in files:
    path = os.path.join(folder_path, file)
    df = pd.read_sas(path)
    # Opcional: Convertir SEQN a entero para evitar problemas de tipos
    df['SEQN'] = df['SEQN'].astype(int)
    datasets.append(df)
    
print("--- Columnas por Dataset ---")
for nombre_archivo, df in zip(files, datasets):
    print(f"\nArchivo: {nombre_archivo}")
    print(f"Total columnas: {len(df.columns)}")
    print(list(df.columns))
    print("-" * 30)  

--- Columnas por Dataset ---

Archivo: ALB_CR_L.xpt
Total columnas: 8
['SEQN', 'URXUMA', 'URXUMS', 'URDUMALC', 'URXUCR', 'URXCRS', 'URDUCRLC', 'URDACT']
------------------------------

Archivo: CBC_L.xpt
Total columnas: 23
['SEQN', 'WTPH2YR', 'LBXWBCSI', 'LBXLYPCT', 'LBXMOPCT', 'LBXNEPCT', 'LBXEOPCT', 'LBXBAPCT', 'LBDLYMNO', 'LBDMONO', 'LBDNENO', 'LBDEONO', 'LBDBANO', 'LBXRBCSI', 'LBXHGB', 'LBXHCT', 'LBXMCVSI', 'LBXMC', 'LBXMCHSI', 'LBXRDW', 'LBXPLTSI', 'LBXMPSI', 'LBXNRBC']
------------------------------

Archivo: TRIGLY_L.xpt
Total columnas: 10
['SEQN', 'WTSAF2YR', 'LBXTLG', 'LBDTRSI', 'LBDLDL', 'LBDLDLSI', 'LBDLDLM', 'LBDLDMSI', 'LBDLDLN', 'LBDLDNSI']
------------------------------

Archivo: HDL_L.xpt
Total columnas: 4
['SEQN', 'WTPH2YR', 'LBDHDD', 'LBDHDDSI']
------------------------------

Archivo: HSCRP_L.xpt
Total columnas: 4
['SEQN', 'WTPH2YR', 'LBXHSCRP', 'LBDHRPLC']
------------------------------

Archivo: PBCD_L.xpt
Total columnas: 17
['SEQN', 'WTPH2YR', 'LBXBPB', 'LBDBPBSI'

In [11]:
# 2. Proceso de Merge evitando duplicados
# Empezamos con el primer dataset completo
merged_df = datasets[0]

for i in range(1, len(datasets)):
    df_actual = datasets[i]
    
    # Buscamos qué columnas del dataset actual NO están todavía en merged_df
    # Pero mantenemos 'SEQN' que es nuestra llave de unión
    columnas_nuevas = [col for col in df_actual.columns if col not in merged_df.columns]
    columnas_a_mantener = ['SEQN'] + columnas_nuevas
    
    # Hacemos el merge solo con las columnas que no se repiten
    merged_df = pd.merge(
        merged_df, 
        df_actual[columnas_a_mantener], 
        on='SEQN', 
        how='inner'
    )

# 3. Verificación final
print(f"Merge finalizado con {len(merged_df)} filas.")
print(f"Total de columnas únicas obtenidas: {len(merged_df.columns)}")

# 4. Análisis de nulos rápido
print("\n--- Conteo de nulos en las primeras 5 columnas ---")
print(merged_df.isnull().sum().head(20))

# 5. Guardar el resultado
merged_df.to_csv('data/merged_sets/merged_lab_data.csv', index=False)
print("\nArchivo guardado exitosamente como 'merged_lab_data.csv'")

Merge finalizado con 3996 filas.
Total de columnas únicas obtenidas: 58

--- Conteo de nulos en las primeras 5 columnas ---
SEQN          0
URXUMA      107
URXUMS      107
URDUMALC    107
URXUCR      107
URXCRS      107
URDUCRLC    107
URDACT      107
WTPH2YR       0
LBXWBCSI    258
LBXLYPCT    264
LBXMOPCT    264
LBXNEPCT    264
LBXEOPCT    264
LBXBAPCT    264
LBDLYMNO    264
LBDMONO     264
LBDNENO     264
LBDEONO     264
LBDBANO     264
dtype: int64

Archivo guardado exitosamente como 'merged_lab_data.csv'


In [12]:
# 1. Cargar el merge generado anteriormente
df = pd.read_csv('merged_lab_data.csv')

# 2. Definir el umbral (e.g., conservar columnas con al menos 70% de datos)
umbral_porcentaje = 0.7
min_datos_requeridos = int(umbral_porcentaje * len(df))

# 3. Aplicar la limpieza
# axis=1 indica que eliminamos columnas. thresh pide el mínimo de valores NO nulos.
df_clean = df.dropna(axis=1, thresh=min_datos_requeridos)

print(f"Columnas originales: {df.shape[1]}")
print(f"Columnas después de limpieza: {df_clean.shape[1]}")
print(f"Variables eliminadas por exceso de nulos: {df.shape[1] - df_clean.shape[1]}")

# 4. Guardar el dataset optimizado
df_clean.to_csv('data/merged_sets/merged_lab_data_clean.csv', index=False)

# Mostrar las primeras 5 columnas restantes para verificar
print("\nPrimeras columnas del dataset limpio:")
print(df_clean.columns.tolist()[:10])

Columnas originales: 58
Columnas después de limpieza: 58
Variables eliminadas por exceso de nulos: 0

Primeras columnas del dataset limpio:
['SEQN', 'URXUMA', 'URXUMS', 'URDUMALC', 'URXUCR', 'URXCRS', 'URDUCRLC', 'URDACT', 'WTPH2YR', 'LBXWBCSI']


## Convert Dietary into csv

En este caso sólo poseíamos un único set por lo que de momento no vamos a realizar ningún merge

In [14]:
# 1. Cargar el archivo dietético
file_path = 'data/DSQTOT_L.xpt'
df_diet = pd.read_sas(file_path)

# Asegurar que SEQN sea entero
df_diet['SEQN'] = df_diet['SEQN'].astype(int)

# 2. Análisis de integridad previo
total_filas = len(df_diet)
nulos_por_columna = df_diet.isnull().sum()

# 3. Limpieza por umbral (Conservar columnas con al menos 70% de datos)
umbral = 0.7
min_datos = int(umbral * total_filas)
df_diet_clean = df_diet.dropna(axis=1, thresh=min_datos)

# 4. Mostrar balance de la limpieza
print(f"--- Resumen de carga: DSQTOT ---")
print(f"Registros totales: {total_filas}")
print(f"Columnas originales: {df_diet.shape[1]}")
print(f"Columnas tras limpieza (>{umbral*100}% datos): {df_diet_clean.shape[1]}")

# 5. Guardar como CSV
df_diet_clean.to_csv('data/merged_sets/dietary_data.csv', index=False)
print("\nArchivo 'dietary_data.csv' generado exitosamente.")

--- Resumen de carga: DSQTOT ---
Registros totales: 8860
Columnas originales: 40
Columnas tras limpieza (>70.0% datos): 6

Archivo 'dietary_data.csv' generado exitosamente.


## Convert Demography into csv

En este caso sólo poseíamos un único set por lo que de momento no vamos a realizar ningún merge

In [15]:
# 1. Cargar el archivo dietético
file_path = 'data/DEMO_L.xpt'
df_diet = pd.read_sas(file_path)

# Asegurar que SEQN sea entero
df_diet['SEQN'] = df_diet['SEQN'].astype(int)

# 2. Análisis de integridad previo
total_filas = len(df_diet)
nulos_por_columna = df_diet.isnull().sum()

# 3. Limpieza por umbral (Conservar columnas con al menos 70% de datos)
umbral = 0.7
min_datos = int(umbral * total_filas)
df_diet_clean = df_diet.dropna(axis=1, thresh=min_datos)

# 4. Mostrar balance de la limpieza
print(f"--- Resumen de carga: DSQTOT ---")
print(f"Registros totales: {total_filas}")
print(f"Columnas originales: {df_diet.shape[1]}")
print(f"Columnas tras limpieza (>{umbral*100}% datos): {df_diet_clean.shape[1]}")

# 5. Guardar como CSV
df_diet_clean.to_csv('data/merged_sets/demographic_data.csv', index=False)
print("\nArchivo 'demographic_data.csv' generado exitosamente.")

--- Resumen de carga: DSQTOT ---
Registros totales: 11933
Columnas originales: 27
Columnas tras limpieza (>70.0% datos): 15

Archivo 'demographic_data.csv' generado exitosamente.


### Revisión de los datasets generados 

Antes de proceder a unir los cuatro sets, vamos a pomprobar el tamaño de los mismos para asegurarnos que todos poseen una cantidad de pacientes adecuada para el merge de todas.

In [3]:
folder_path = 'data/merged_sets/'
files = [
    "demographic_data.csv",
    "dietary_data.csv",
    "merged_examination_data_clean.csv",
    "merged_lab_data_clean.csv"
]

print(f"{'Archivo':<35} | {'Filas':<8} | {'Cols':<5} | {'Tamaño disco':<12}")
print("-" * 70)

for file_name in files:
    path = os.path.join(folder_path, file_name)
    
    if os.path.exists(path):
        # Cargamos solo una parte o el archivo completo
        df_temp = pd.read_csv(path)
        
        # Info del DataFrame
        rows, cols = df_temp.shape
        
        print(f"{file_name:<35} | {rows:<8} | {cols:<5}")
    else:
        print(f"{file_name:<35} | No encontrado")

Archivo                             | Filas    | Cols  | Tamaño disco
----------------------------------------------------------------------
demographic_data.csv                | 11933    | 15   
dietary_data.csv                    | 8860     | 6    
merged_examination_data_clean.csv   | 4771     | 49   
merged_lab_data_clean.csv           | 3996     | 58   


In [5]:
for file_name in files:
    path = os.path.join(folder_path, file_name)
    
    if os.path.exists(path):
        # Cargamos solo la cabecera (header) para que sea ultra rápido
        # nrows=0 evita cargar miles de filas innecesariamente
        df_cols = pd.read_csv(path, nrows=0)
        
        print(f"\n📂 ARCHIVO: {file_name}")
        print(f"{'='*len(file_name) + '==========='}")
        print(f"Total de columnas: {len(df_cols.columns)}")
        print(list(df_cols.columns))
        print("-" * 50)
    else:
        print(f"\n⚠️ El archivo {file_name} no se encontró en la ruta.")


📂 ARCHIVO: demographic_data.csv
Total de columnas: 15
['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'DMDBORN4', 'DMDHHSIZ', 'WTINT2YR', 'WTMEC2YR', 'SDMVSTRA', 'SDMVPSU', 'INDFMPIR']
--------------------------------------------------

📂 ARCHIVO: dietary_data.csv
Total de columnas: 6
['SEQN', 'WTDRD1', 'DSDCOUNT', 'DSDANCNT', 'DSD010', 'DSD010AN']
--------------------------------------------------

📂 ARCHIVO: merged_examination_data_clean.csv
Total de columnas: 49
['SEQN', 'BAXMSTAT', 'BAX5STAT', 'BAQ110', 'BAQ121', 'BAQ125', 'BAQ132', 'BAQ140', 'BAQ150', 'BAQ160', 'BAQ170', 'BAQ173', 'BAXPF11', 'BAXTC11', 'BAXPF21', 'BAXTC21', 'BAXPF31', 'BAXTC31', 'BAXPF41', 'BAXTC41', 'BPAOARM', 'BPAOCSZ', 'BPXOSY1', 'BPXODI1', 'BPXOSY2', 'BPXODI2', 'BPXOSY3', 'BPXODI3', 'BPXOPLS1', 'BPXOPLS2', 'BPXOPLS3', 'BMDSTATS', 'BMXWT', 'BMXHT', 'BMXBMI', 'BMXLEG', 'BMXARML', 'BMXARMC', 'BMXWAIST', 'BMXHIP', 'LUAXSTAT', 'LUAPNME', 'LUANMVGP', 'LUANMTGP', 'LUXSMED